#Neural Networks

In [ ]:
import os, random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
df = pd.read_csv("energydata_complete.csv")
print(df.shape)
print(df.columns)
df.head()

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month

df = df.drop(columns=["date", "rv1", "rv2"], errors="ignore")

y = df["Appliances"].copy()
X = df.drop(columns=["Appliances"])

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

In [ ]:
def build_model(input_dim, units=(256,128,64,32), dropout=0.2, lr=1e-3):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for u in units:
        model.add(layers.Dense(u, activation="relu"))
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.Huber(),
        metrics=[keras.metrics.MeanAbsoluteError()]
    )
    return model

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)
]

model = build_model(input_dim=X_train_s.shape[1])
history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
pred = model.predict(X_test_s).ravel()

mae = mean_absolute_error(y_test, pred)
mse = mean_squared_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"Test MAE: {mae:.3f}")
print(f"Test MSE: {mse:.3f}")
print(f"Test R2 : {r2:.3f}")